<a href="https://colab.research.google.com/github/gmauricio-toledo/math-ML/blob/main/Notebooks/18-Redes_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificación de MNIST: Comparativa Keras vs. PyTorch

En este notebook entrenaremos una red neuronal MLP (Multilayer Perceptron) sencilla e idéntica para clasificar dígitos escritos a mano usando **TensorFlow/Keras** y **PyTorch**.

## 1. Preparación de Datos
Cargamos el dataset completo de MNIST y lo preparamos para ambos frameworks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

# Cargar el dataset completo y dividir en train/test
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

# Normalizar las imágenes a rango [0, 1] y aplanar las imágenes de 28x28 a vectores de 784 elementos
X_train = X_train_raw.reshape(-1, 28 * 28).astype('float32') / 255.0
X_test = X_test_raw.reshape(-1, 28 * 28).astype('float32') / 255.0

y_train = y_train_raw.astype('int64')
y_test = y_test_raw.astype('int64')

print(f"Datos de Entrenamiento: {X_train.shape}, Etiquetas: {y_train.shape}")
print(f"Datos de Prueba: {X_test.shape}, Etiquetas: {y_test.shape}")

## 2. Sección Keras / TensorFlow
Creamos un modelo secuencial, imprimimos su resumen de parámetros, lo entrenamos, graficamos las curvas de aprendizaje y guardamos el modelo.

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

# 1. Definir la arquitectura de la red (MLP sencillo)
keras_model = Sequential([
    Dense(128, activation='relu', input_shape=(784,)),
    Dense(10, activation='softmax')
])

# 2. Imprimir arquitectura y parámetros
keras_model.summary()

# 3. Compilar el modelo
keras_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# 4. Entrenar la red
history_keras = keras_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

# 5. Guardar el modelo
keras_model.save('mnist_keras_model.h5')

In [ ]:
# Graficar Resultados de Keras
plt.figure(figsize=(12, 4))

# Gráfico de Pérdida (Loss)
plt.subplot(1, 2, 1)
plt.plot(history_keras.history['loss'], label='Train Loss')
plt.plot(history_keras.history['val_loss'], label='Val Loss')
plt.title('Keras: Pérdida (Loss)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Gráfico de Precisión (Accuracy)
plt.subplot(1, 2, 2)
plt.plot(history_keras.history['accuracy'], label='Train Acc')
plt.plot(history_keras.history['val_accuracy'], label='Val Acc')
plt.title('Keras: Precisión (Accuracy)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 3. Sección PyTorch
Construimos exactamente el mismo modelo y ciclo de entrenamiento con PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Convertir datos de NumPy a tensores de PyTorch
torch_X_train = torch.tensor(X_train)
torch_y_train = torch.tensor(y_train)
torch_X_test = torch.tensor(X_test)
torch_y_test = torch.tensor(y_test)

# Crear DataLoaders para facilitar el entrenamiento por lotes (batches)
train_dataset = TensorDataset(torch_X_train, torch_y_train)
test_dataset = TensorDataset(torch_X_test, torch_y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)



In [ ]:
# 1. Definir la arquitectura (Secuencial equivalente a Keras)
pytorch_model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

# 2. Imprimir arquitectura y contar parámetros
print(pytorch_model)
total_params = sum(p.numel() for p in pytorch_model.parameters() if p.requires_grad)
print(f"Total de parámetros entrenables: {total_params:,}\n")

# 3. Definir optimizador y función de pérdida
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pytorch_model.parameters())

# Historiales para graficar
history_pytorch = {'loss': [], 'val_loss': [], 'acc': [], 'val_acc': []}

In [ ]:
# 4. Bucle/Ciclo de entrenamiento sencillo

epochs = 10 # Número de veces que pasaremos por todo el dataset

for epoch in range(epochs): # Iterar sobre cada epoch
    pytorch_model.train() # Poner el modelo en modo entrenamiento (activa Dropout, BatchNorm, etc.)
    running_loss = 0.0 # Variable para acumular la pérdida del entrenamiento
    correct_train = 0 # Variable para contar las predicciones correctas
    total_train = 0 # Variable para contar el total de ejemplos procesados

    for inputs, targets in train_loader: # Iterar sobre lotes (batches) de datos de entrenamiento
        optimizer.zero_grad() # Limpiar los gradientes acumulados del lote anterior
        outputs = pytorch_model(inputs) # Pasar los datos por el modelo (Forward pass)
        loss = criterion(outputs, targets) # Calcular el error (pérdida) de las predicciones
        loss.backward() # Calcular los gradientes de la pérdida (Backward pass)
        optimizer.step() # Actualizar los pesos del modelo usando los gradientes

        running_loss += loss.item() * inputs.size(0) # Acumular la pérdida total del lote
        _, predicted = outputs.max(1) # Obtener la clase con mayor probabilidad
        total_train += targets.size(0) # Sumar la cantidad de ejemplos del lote al total
        correct_train += predicted.eq(targets).sum().item() # Sumar cuántas predicciones fueron correctas

    epoch_loss = running_loss / len(train_dataset) # Calcular la pérdida media del epoch
    epoch_acc = correct_train / total_train # Calcular la precisión media del epoch

    # Validación al final del epoch
    pytorch_model.eval() # Poner el modelo en modo evaluación (desactiva Dropout, etc.)
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad(): # Desactivar el cálculo de gradientes para ahorrar memoria y tiempo
        for inputs, targets in test_loader: # Iterar sobre lotes de datos de prueba
            outputs = pytorch_model(inputs) # Predicciones del modelo
            loss = criterion(outputs, targets) # Calcular error de validación
            val_loss += loss.item() * inputs.size(0) # Acumular pérdida de validación
            _, predicted = outputs.max(1) # Obtener clases predichas
            total_val += targets.size(0) # Acumular total de ejemplos de validación
            correct_val += predicted.eq(targets).sum().item() # Acumular aciertos de validación

    epoch_val_loss = val_loss / len(test_dataset) # Pérdida media de validación
    epoch_val_acc = correct_val / total_val # Precisión media de validación

    # Guardar en el historial
    history_pytorch['loss'].append(epoch_loss)
    history_pytorch['acc'].append(epoch_acc)
    history_pytorch['val_loss'].append(epoch_val_loss)
    history_pytorch['val_acc'].append(epoch_val_acc)

    print(f"Epoch {epoch+1:02d}/{epochs:02d} -> Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f} | Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.4f}")

# 5. Guardar el modelo
torch.save(pytorch_model.state_dict(), 'mnist_pytorch_model.pth')

In [ ]:
# Graficar Resultados de PyTorch
plt.figure(figsize=(12, 4))

# Gráfico de Pérdida (Loss)
plt.subplot(1, 2, 1)
plt.plot(history_pytorch['loss'], label='Train Loss')
plt.plot(history_pytorch['val_loss'], label='Val Loss')
plt.title('PyTorch: Pérdida (Loss)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Gráfico de Precisión (Accuracy)
plt.subplot(1, 2, 2)
plt.plot(history_pytorch['acc'], label='Train Acc')
plt.plot(history_pytorch['val_acc'], label='Val Acc')
plt.title('PyTorch: Precisión (Accuracy)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# 🔴 Trabajo en clase

A continuación se presentan las definiciones de diversas funciones de activación utilizadas comúnmente en redes neuronales. Para cada una de ellas, realiza los ejercicios de derivación propuestos.

* **Función Sigmoide (Logística)**: La función sigmoide se utiliza frecuentemente para transformar valores al rango abierto $(0, 1)$, interpretándose a menudo como una probabilidad. Su definición matemática es:

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$


Calcula la primera derivada $\frac{d\sigma}{dx}$ utilizando las reglas de derivación habituales y demuestra algebraicamente que el resultado se puede expresar en términos de la propia función de la siguiente manera:

$$\frac{d\sigma}{dx} = \sigma(x)(1 - \sigma(x))$$

- **Función Tangente Hiperbólica ($\tanh$)**: La función tangente hiperbólica es una versión escalada y desplazada de la sigmoide que mapea los valores de entrada al rango abierto $(-1, 1)$, manteniendo una media cercana a cero que facilita la convergencia durante el entrenamiento. Su definición matemática es:

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

Calcula la primera derivada $\frac{d}{dx}\tanh(x)$ y demuestra que el resultado se puede expresar en términos de la propia función de la siguiente manera:

$$\frac{d}{dx}\tanh(x) = 1 - \tanh^2(x)$$

- **Función Swish (SiLU)**: La función Swish es una activación suave y no monótona que combina la entrada lineal con la función sigmoide, permitiendo el flujo de gradientes pequeños en el dominio negativo.

$$\text{Swish}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

Calcula la primera derivada $\frac{d}{dx}\text{Swish}(x)$ y demuestra que el resultado se puede expresar en términos de la función sigmoide y de la propia función Swish como:

$$\frac{d}{dx}\text{Swish}(x) = \text{Swish}(x) + \sigma(x)(1 - \text{Swish}(x))$$

- **Función ELU (Exponential Linear Unit)**: La función ELU reduce la presencia de gradientes desvanecidos al incorporar un término exponencial para valores negativos, ajustado por una constante $\alpha > 0$. Para la región negativa ($x \le 0$), su definición es:

$$\text{ELU}(x; \alpha) = \alpha (e^x - 1) \quad \text{para } x \le 0$$

Calcula la derivada $\frac{d}{dx}\text{ELU}(x; \alpha)$ para el dominio $x \le 0$ y demuestra que se puede expresar en función del valor de la propia curva como:

$$\frac{d}{dx}\text{ELU}(x; \alpha) = \text{ELU}(x; \alpha) + \alpha$$


---

En cada caso, grafica cada función de activación usando `matplotlib`.

- **Colapso Lineal por Ausencia de Funciones de Activación**: En una red neuronal profunda, la omisión de funciones de activación no lineales (es decir, el uso de la función identidad $\sigma(z) = z$) provoca que la composición de transformaciones afines en capas sucesivas se reduzca algebraicamente a una única transformación lineal.

  Considera una red neuronal *feedforward* con la siguiente arquitectura de dimensiones:
  - **Entrada**: Vector $x \in \mathbb{R}^2$.
  - **Capa Oculta 1**: $3$ neuronas.
  - **Capa Oculta 2**: $2$ neuronas.
  - **Capa de Salida**: $1$ neurona.

  En todas las capas, la función de activación es $\sigma(z) = z$.

  **Instrucciones del Ejercicio:**

  a) Obten la expresión para la salida de la red en términos de la entrada $x=(x_1,x_2) \in \mathbb{R}^2$.

  b) Demuestra que la red neuronal completa es equivalente a un modelo de regresión lineal simple de la forma:
  $$\hat{y} = W_{eq}x + b_{eq}$$
  Escribe las fórmulas explícitas para la matriz de pesos equivalente $W_{eq}$ y el sesgo equivalente $b_{eq}$ en términos de las matrices y/o pesos intermedios.
